# Ginzburg–Landau — dataset generation

Generate the shared MATLAB dataset for both SLD tasks. The supplied defaults produce ten initial conditions, each with HR `uv` shape `(2, 1200, 256, 256)` and clean LR shape `(2, 300, 32, 32)`. Temporal refinement is 4× and spatial refinement is 8×.

Run the configuration cell, then the generation cell. The last cell is an optional visualization of the generated IC 1 dataset. The original PDE, RK4 solver, numerical settings and seeds are preserved. Generation writes to the configured data directory and creates visualization files in its results directory.

Install `pyrtools` for the supplied blurDn downsampling path. The original Gaussian fallback is retained if it is unavailable; that fallback uses a different downsampling operator. Dependencies: NumPy, SciPy, Matplotlib and pyrtools. Original file: `GL data (1).ipynb`.


In [ ]:
from pathlib import Path
import os

# Start Jupyter in the repository or any directory beneath it.
def _find_repository_root():
    working_dir = Path.cwd().resolve()
    for candidate in (working_dir, *working_dir.parents):
        if (candidate / "README.md").is_file() and (candidate / "data_generation").is_dir():
            return candidate
    raise RuntimeError("Cannot find the SLD repository. Start Jupyter from the repository directory.")

ROOT = _find_repository_root()
DATA_ROOT = Path(os.environ.get("SLD_DATA_ROOT", str(ROOT / "data"))).expanduser()
RESULTS_ROOT = Path(os.environ.get("SLD_RESULTS_ROOT", str(ROOT / "results"))).expanduser()
if not DATA_ROOT.is_absolute():
    DATA_ROOT = ROOT / DATA_ROOT
if not RESULTS_ROOT.is_absolute():
    RESULTS_ROOT = ROOT / RESULTS_ROOT
DATA_DIR = DATA_ROOT / "ginzburg_landau"
RESULTS_DIR = RESULTS_ROOT / "data_generation" / "ginzburg_landau"
print(f"Repository: {ROOT}")
print(f"Data: {DATA_DIR}")
print(f"Results: {RESULTS_DIR}")


In [ ]:
import numpy as np
import scipy.io as scio
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

try:
    import pyrtools as pt
    HAS_PYRTOOLS = True
except ImportError:
    print("Warning: pyrtools not installed. Will use gaussian blur + decimation.")
    HAS_PYRTOOLS = False


# =========================
# Create directories
# =========================
data_dir = DATA_DIR
fig_dir = RESULTS_DIR / 'figures'

data_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(1)


# =========================
# Numerical operators
# =========================
def apply_laplacian(mat, dx):
    """
    4th-order finite-difference Laplacian in periodic BC,
    matching the style used in your 2D GS code.
    """
    neigh_mat = -5 * mat.copy()

    neighbors = [
        (4/3,   (-1, 0)),
        (4/3,   ( 1, 0)),
        (4/3,   ( 0,-1)),
        (4/3,   ( 0, 1)),
        (-1/12, (-2, 0)),
        (-1/12, ( 2, 0)),
        (-1/12, ( 0,-2)),
        (-1/12, ( 0, 2)),
    ]

    for weight, neigh in neighbors:
        neigh_mat += weight * np.roll(mat, shift=neigh, axis=(0, 1))

    return neigh_mat / dx**2


def gl_rhs(u, v, eps, nu, gamma_re, gamma_im, dx):
    """
    RHS of 2D Ginzburg-Landau system.
    """
    lap_u = apply_laplacian(u, dx)
    lap_v = apply_laplacian(v, dx)

    amp2 = u**2 + v**2

    rhs_u = eps * lap_u + nu * u - amp2 * (gamma_re * u - gamma_im * v)
    rhs_v = eps * lap_v + nu * v - amp2 * (gamma_re * v + gamma_im * u)

    return rhs_u, rhs_v


def update_rk4_gl(u0, v0, eps, nu, gamma_re, gamma_im, delta_t, dx):
    """
    RK4 update for 2D GL.
    """
    # Stage 1
    k1_u, k1_v = gl_rhs(u0, v0, eps, nu, gamma_re, gamma_im, dx)

    # Stage 2
    u1 = u0 + 0.5 * delta_t * k1_u
    v1 = v0 + 0.5 * delta_t * k1_v
    k2_u, k2_v = gl_rhs(u1, v1, eps, nu, gamma_re, gamma_im, dx)

    # Stage 3
    u2 = u0 + 0.5 * delta_t * k2_u
    v2 = v0 + 0.5 * delta_t * k2_v
    k3_u, k3_v = gl_rhs(u2, v2, eps, nu, gamma_re, gamma_im, dx)

    # Stage 4
    u3 = u0 + delta_t * k3_u
    v3 = v0 + delta_t * k3_v
    k4_u, k4_v = gl_rhs(u3, v3, eps, nu, gamma_re, gamma_im, dx)

    u = u0 + delta_t * (k1_u + 2*k2_u + 2*k3_u + k4_u) / 6.0
    v = v0 + delta_t * (k1_v + 2*k2_v + 2*k3_v + k4_v) / 6.0

    return u, v


# =========================
# Initial condition
# =========================
def get_initial_u_and_v(N, random_influence=0.01, ic_seed=1):
    """
    Initial condition from the paper:
        A0(x, y) = 10 (y + i x) exp(-25(x^2 + y^2))
    so
        u0 = 10 y exp(...)
        v0 = 10 x exp(...)
    
    Add small noise controlled by ic_seed to create slightly different ICs.
    """
    np.random.seed(ic_seed)

    x = np.linspace(-1, 1, N, endpoint=False)
    y = np.linspace(-1, 1, N, endpoint=False)
    xx, yy = np.meshgrid(x, y, indexing='ij')

    gaussian = np.exp(-25 * (xx**2 + yy**2))
    u = 10.0 * yy * gaussian
    v = 10.0 * xx * gaussian

    if random_influence > 0:
        u += random_influence * np.random.randn(N, N)
        v += random_influence * np.random.randn(N, N)

    return u.astype(np.float32), v.astype(np.float32)


# =========================
# Visualization
# =========================
def postProcess(output, N, xmin, xmax, ymin, ymax, num, batch, save_dir):
    """
    Visualization function.
    output: [2, T, H, W] or [T, 2, H, W]
    """
    x = np.linspace(xmin, xmax, N+1)[:-1]
    y = np.linspace(ymin, ymax, N+1)[:-1]
    x_star, y_star = np.meshgrid(x, y)

    if output.shape[0] == 2:
        u_pred = output[0, num, :, :]
        v_pred = output[1, num, :, :]
    else:
        u_pred = output[num, 0, :, :]
        v_pred = output[num, 1, :, :]

    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(6, 3))
    fig.subplots_adjust(hspace=0.3, wspace=0.3)

    cf = ax[0].scatter(x_star, y_star, c=u_pred, alpha=0.95, edgecolors='none',
                       cmap='RdBu_r', marker='s', s=2)
    ax[0].axis('square')
    ax[0].set_xlim([xmin, xmax])
    ax[0].set_ylim([ymin, ymax])
    ax[0].set_title('u-FDM')
    fig.colorbar(cf, ax=ax[0])

    cf = ax[1].scatter(x_star, y_star, c=v_pred, alpha=0.95, edgecolors='none',
                       cmap='RdBu_r', marker='s', s=2)
    ax[1].axis('square')
    ax[1].set_xlim([xmin, xmax])
    ax[1].set_ylim([ymin, ymax])
    ax[1].set_title('v-FDM')
    fig.colorbar(cf, ax=ax[1])

    plt.savefig(save_dir / f'uv_[b={batch}][t={num}].png', dpi=150, bbox_inches='tight')
    plt.close('all')


# =========================
# Downsampling
# =========================
def downsample_temporal(data, t_ds=4):
    """
    data: [2, T, H, W]
    """
    return data[:, ::t_ds, :, :]


def downsample_spatial_gaussian(data_high, ds=16, sigma=2.0):
    """
    Spatial anti-aliased downsampling for [2, T, H, W].
    Blur each frame, then decimate.
    For the current 256 -> 32 configuration, ds = 8.
    """
    C, T, H, W = data_high.shape
    assert H % ds == 0 and W % ds == 0

    h_lr, w_lr = H // ds, W // ds
    data_low = np.zeros((C, T, h_lr, w_lr), dtype=np.float32)

    for c in range(C):
        for t in range(T):
            blurred = gaussian_filter(data_high[c, t], sigma=sigma, mode='wrap')
            data_low[c, t] = blurred[::ds, ::ds]

    return data_low


def downsample_spatial_official_pyrtools(data_high, n_levels=3):
    """
    Repeated blurDn with pyrtools.
    Input: [2, T, H, W]
    Output: [2, T, H/2^n_levels, W/2^n_levels]
    """
    if not HAS_PYRTOOLS:
        print("pyrtools not installed. Falling back to gaussian blur + decimation.")
        ds = 2 ** n_levels
        return downsample_spatial_gaussian(data_high, ds=ds, sigma=2.0)

    C, T, H, W = data_high.shape
    h_lr = H // (2 ** n_levels)
    w_lr = W // (2 ** n_levels)
    data_low = np.zeros([C, T, h_lr, w_lr], dtype=np.float32)

    for c in range(C):
        for t in range(T):
            data_low[c, t] = pt.blurDn(data_high[c, t], n_levels=n_levels, filt='binom5')

    return data_low.astype(np.float32)



# =========================
# Observation noise on LR
# =========================
def add_observation_noise_lr(UV_low_clean, noise_level=0.10, noise_type="gaussian",
                             seed=None, per_channel=True, clip=None):
    """
    Add observation noise to LR only.
    UV_low_clean: [2, T, H, W]
    """
    rng = np.random.default_rng(seed)
    UV = UV_low_clean.astype(np.float32, copy=True)

    if noise_type.lower() != "gaussian":
        raise ValueError(f"Unsupported noise_type: {noise_type}")

    if per_channel:
        ch_std = UV.reshape(2, -1).std(axis=1)
        sigma = (noise_level * ch_std).reshape(2, 1, 1, 1)
    else:
        sigma = noise_level * UV.std()

    noise = rng.normal(loc=0.0, scale=1.0, size=UV.shape).astype(np.float32)
    UV_noisy = UV + noise * sigma

    if clip is not None:
        UV_noisy = np.clip(UV_noisy, clip[0], clip[1])

    return UV_noisy


# =========================
# Dataset generation
# =========================
def generate_dataset(ic_number,
                     N=256,
                     T_hr_target=1200,
                     random_influence=0.01,
                     use_official_downsample=True,
                     add_obs_noise=False,
                     obs_noise_level=0.10,
                     obs_noise_seed_base=12345,
                     obs_noise_per_channel=True,
                     save_noisy_lr=True):


    # PDE parameters from the paper
    eps = 0.004
    nu = 10.0
    gamma_re = 10.0
    gamma_im = 15.0

    # Domain
    x_min, x_max = -1.0, 1.0
    dx = (x_max - x_min) / N

    # Time stepping:
    # Use a smaller dt for stability than GS.
    delta_t = 2.0e-4

    # Store T_hr_target HR frames:
    # initial frame + (T_hr_target - 1) saved steps
    # Save every `save_every` internal steps.
    save_every = 84
    N_simulation_steps = (T_hr_target - 1) * save_every

    TEMPORAL_DS = 4
    SPATIAL_DS = 8
    N_LEVELS = 3  # 256 -> 32
    T_lr_target = (T_hr_target + TEMPORAL_DS - 1) // TEMPORAL_DS  # 对1200正好是300
    H_lr = N // SPATIAL_DS
    W_lr = N // SPATIAL_DS


    ri_tag = f"ri{int(random_influence * 1000):03d}"  # e.g. 0.01 -> ri010

    print(f"\n{'='*60}")
    print(f"Generating 2DGL dataset for IC={ic_number}, random_influence={random_influence} (tag: {ri_tag})")
    print(f"{'='*60}")
    print(f"HR resolution: {N}x{N}")
    print(f"Temporal downsampling: {TEMPORAL_DS}x")
    print(f"Spatial downsampling: {SPATIAL_DS}x")
    print(f"Internal dt: {delta_t}, save_every: {save_every}")
    print(f"Total internal simulation steps: {N_simulation_steps}")

    u, v = get_initial_u_and_v(N, random_influence=random_influence, ic_seed=ic_number)
    u_record = u.copy()[None, ...]
    v_record = v.copy()[None, ...]

    for step in range(N_simulation_steps):
        u, v = update_rk4_gl(
            u, v,
            eps=eps,
            nu=nu,
            gamma_re=gamma_re,
            gamma_im=gamma_im,
            delta_t=delta_t,
            dx=dx
        )

        if (step + 1) % save_every == 0:
            if step % 5000 == 0:
                print(f"  Progress: {step+1}/{N_simulation_steps}")
            u_record = np.concatenate((u_record, u[None, ...]), axis=0)
            v_record = np.concatenate((v_record, v[None, ...]), axis=0)

    print(f"  Simulation complete: {u_record.shape[0]} timesteps")

    UV_high = np.stack([u_record, v_record], axis=0).astype(np.float32)  # [2, T, 256, 256]

    save_path_high = data_dir / f'2DGL_IC{ic_number}_{ri_tag}_2x{T_hr_target}x{N}x{N}.mat'
    scio.savemat(save_path_high, {'uv': UV_high})
    print(f"  ✓ Saved HR: {save_path_high.name} | shape: {UV_high.shape}")

    # Downsample
    UV_temp = downsample_temporal(UV_high, t_ds=TEMPORAL_DS)
    print(f"  After temporal DS: {UV_temp.shape}")

    if use_official_downsample and HAS_PYRTOOLS:
        UV_low = downsample_spatial_official_pyrtools(UV_temp, n_levels=N_LEVELS)
    else:
        UV_low = downsample_spatial_gaussian(UV_temp, ds=SPATIAL_DS, sigma=2.0)

    print(f"  After spatial DS: {UV_low.shape}")

    save_path_low = data_dir / f'2DGL_IC{ic_number}_{ri_tag}_2x{UV_low.shape[1]}x{H_lr}x{W_lr}.mat'
    scio.savemat(save_path_low, {'uv': UV_low})
    print(f"  ✓ Saved LR: {save_path_low.name} | shape: {UV_low.shape}")

    # Noisy LR
    if add_obs_noise and save_noisy_lr:
        obsn_tag = f"obsn{int(obs_noise_level * 100):02d}"
        noise_seed = int(obs_noise_seed_base + ic_number)

        UV_low_noisy = add_observation_noise_lr(
            UV_low_clean=UV_low,
            noise_level=obs_noise_level,
            noise_type="gaussian",
            seed=noise_seed,
            per_channel=obs_noise_per_channel,
            clip=None
        )

        save_path_low_noisy = data_dir / f'2DGL_IC{ic_number}_{ri_tag}_{obsn_tag}_2x{UV_low_noisy.shape[1]}x{H_lr}x{W_lr}.mat'
        scio.savemat(save_path_low_noisy, {'uv': UV_low_noisy})
        print(f"  ✓ Saved LR (noisy): {save_path_low_noisy.name} | shape: {UV_low_noisy.shape} | seed: {noise_seed}")

    # Verification
    assert UV_high.shape == (2, T_hr_target, N, N), f"HR shape mismatch: {UV_high.shape}"
    assert UV_low.shape == (2, T_lr_target, H_lr, W_lr), f"LR shape mismatch: {UV_low.shape}"
    print("  ✓ Shape verification passed")

    # Optional visualization
    if ic_number == 1:
        print("  Generating visualizations...")
        for t_idx in [0, 500, 1000, 1500, 2000, 2500, 3000]:
            if t_idx < UV_high.shape[1]:
                postProcess(UV_high, N, x_min, x_max, x_min, x_max, t_idx, ic_number, fig_dir)

    return UV_high, UV_low


# =========================
# Main execution
# =========================
if __name__ == '__main__':
    print("=" * 60)
    print("2D Ginzburg-Landau Dataset Generation (SR x8, Temporal x4)")
    print("=" * 60)
    print(f"Data directory: {data_dir.absolute()}")
    print(f"Figure directory: {fig_dir.absolute()}")

    if not HAS_PYRTOOLS:
        print("\n⚠️  WARNING: pyrtools not installed!")
        print("Will use gaussian blur + decimation for spatial downsampling.\n")

    RANDOM_INFLUENCE = 0.1
    OBS_NOISE_LEVEL = 0.0

    for ic in range(1, 11):
        generate_dataset(
            ic_number=ic,
            N=256,
            T_hr_target=1200,
            random_influence=RANDOM_INFLUENCE,
            use_official_downsample=HAS_PYRTOOLS,
            add_obs_noise=True,
            obs_noise_level=OBS_NOISE_LEVEL,
            obs_noise_seed_base=20250301
        )


    print("\n" + "=" * 60)
    print("2DGL Dataset Generation Complete!")
    print(f"Created {len(list(data_dir.glob('*_2x1200x256x256.mat')))} HR files")
    print(f"Created {len(list(data_dir.glob('*_2x300x32x32.mat')))} LR files")
    print("=" * 60)


## Optional visualization

Run after dataset generation; this cell reads the generated HR data for IC 1.


In [ ]:
import numpy as np
import scipy.io as scio
from pathlib import Path
import matplotlib.pyplot as plt

# =========================
# User settings
# =========================
data_dir = DATA_DIR

ic_number = 1
ri_tag = "ri100"

mat_path = data_dir / f'2DGL_IC{ic_number}_{ri_tag}_2x1200x256x256.mat'

internal_dt = 2.0e-4
save_every = 84
dt_frame = internal_dt * save_every

target_times = [0, 2, 5, 10, 15, 20]

xmin, xmax = -1.0, 1.0
ymin, ymax = -1.0, 1.0


# =========================
# Load data
# =========================
UV = scio.loadmat(mat_path)['uv']   # [2, T, H, W]
C, T_frames, H, W = UV.shape

frame_indices = []
for t in target_times:
    idx = int(round(t / dt_frame))
    idx = max(0, min(idx, T_frames - 1))
    frame_indices.append(idx)

actual_times = [idx * dt_frame for idx in frame_indices]

print("Loaded:", mat_path)
print("UV shape:", UV.shape)
print("dt_frame:", dt_frame)
print("Frame indices:", frame_indices)
print("Actual times:", actual_times)


# =========================
# Plot u and v
# =========================
ncols = len(frame_indices)

fig, axes = plt.subplots(
    2, ncols,
    figsize=(3.2 * ncols, 6.0),
    constrained_layout=True
)

u_imgs = [UV[0, idx] for idx in frame_indices]
v_imgs = [UV[1, idx] for idx in frame_indices]

# u/v 共用一个对称色条范围
absmax = max(
    max(np.abs(img).max() for img in u_imgs),
    max(np.abs(img).max() for img in v_imgs)
)

for j, idx in enumerate(frame_indices):
    t_actual = idx * dt_frame

    im0 = axes[0, j].imshow(
        UV[0, idx].T,
        origin='lower',
        extent=[xmin, xmax, ymin, ymax],
        cmap='RdBu_r',
        vmin=-absmax,
        vmax=absmax,
        interpolation='nearest'
    )
    axes[0, j].set_title(f"u, T = {t_actual:.2f}s")
    axes[0, j].set_xlabel("x")
    axes[0, j].set_ylabel("y")
    axes[0, j].set_aspect("equal")

    im1 = axes[1, j].imshow(
        UV[1, idx].T,
        origin='lower',
        extent=[xmin, xmax, ymin, ymax],
        cmap='RdBu_r',
        vmin=-absmax,
        vmax=absmax,
        interpolation='nearest'
    )
    axes[1, j].set_title(f"v, T = {t_actual:.2f}s")
    axes[1, j].set_xlabel("x")
    axes[1, j].set_ylabel("y")
    axes[1, j].set_aspect("equal")

fig.colorbar(im0, ax=axes, shrink=0.85)
fig.suptitle(f"2D Ginzburg-Landau | IC{ic_number} | HR", fontsize=15)

plt.show()
